# Exercise 4: Option order in TUMLU

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Evaluation and Benchmarks · version 0.1.0*

> Test TUMLU option-order sensitivity on development data, then freeze the scoring protocol before running the audited test subset.

**Setup.** `pip install "stanza==1.11.0" sacrebleu torch transformers sentencepiece llama-cpp-python fasttext-wheel pyarrow requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Exercises 1 and 2 tag the **UD 2.17** Kazakh-KTB and Turkish-IMST test files with Stanza 1.11.0 over the gold words (models from `STANZA_RESOURCES_DIR`, files cached in `UD217_DIR` and checked against their SHA-256 hashes). Exercise 3 uses the X-WMT English–Turkish test set (TIL release, commit `7d988cc`), TurkicNLP's Apertium analyser and generator for Turkish and NLLB-200 distilled 600M (`NLLB_PATH`). Exercises 4 and 5 use Qwen2.5-1.5B-Instruct as an 8-bit GGUF file run with llama.cpp on the CPU (`QWEN_GGUF`, its tokeniser from `QWEN_TOKENIZER`) and GlotLID v3 (`GLOTLID_PATH`). Exercise 4 needs the Kazakh TUMLU-mini files (`TUMLU_DIR`, revision `3523de7`), checked against the hashes of the book's audit. Model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The protocol is the chapter's: the question and four labelled options in Qwen2.5-1.5B-Instruct's chat template, scored by the next-token log-probability of A, B, C and D. The 40 development questions are presented under four cyclic rotations of their options; the scoring policy is chosen there and applied once to the 797 valid test questions.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {   # UD 2.17 release files
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Kazakh-KTB/kk_ktb-ud-train.conllu": "6297597914a3f319db573573f927a6577c702f5fa00dcd653162aab46026aeaa",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
}
UD_DIR = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def ud_path(key):
    path = UD_DIR / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    if sha256(path) != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos feats")
def read(key):
    """Sentences (lists of Word) of a UD file; multiword-token ranges and empty nodes are skipped."""
    sents, words = [], []
    for line in ud_path(key).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[5]))
    return sents

def feats(s):
    return dict(f.split("=", 1) for f in (s or "_").split("|") if "=" in f)

def tag(lang, key):
    """Stanza 1.11 POS/feature/lemma predictions for the gold words of a UD file (cached)."""
    gold = read(key)
    path = CACHE / f"ch15-stanza-{lang}-{SHA256[key][:12]}.json"
    if not path.exists():
        import logging, stanza
        logging.getLogger("stanza").setLevel(logging.ERROR)
        nlp = stanza.Pipeline(lang, processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False,
                              model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())),
                              download_method=None)
        doc = nlp([[w.form for w in s] for s in gold])
        pred = [[[w.lemma or "_", w.upos or "_", w.feats or "_"] for w in s.words] for s in doc.sentences]
        path.write_text(json.dumps(pred, ensure_ascii=False))
    pred = json.loads(path.read_text())
    assert [len(s) for s in pred] == [len(s) for s in gold]
    return gold, [[Word(g.id, g.form, *p) for g, p in zip(gs, ps)] for gs, ps in zip(gold, pred)]

def wilson(k, n, z=1.959963984540054):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---- Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) and GlotLID, for Exercises 4 and 5 ---------------
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
_LLM = {}
def llm():
    if "m" not in _LLM:
        from llama_cpp import Llama
        _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
    return _LLM["m"]

REPLIES = CACHE / "ch15-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["value"]
def cached(key_obj, compute):
    key = hashlib.sha1(json.dumps(key_obj, ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        _CACHED[key] = compute()
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "input": key_obj, "value": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

def chat(messages, max_tokens=120):
    """Greedy reply (cached)."""
    return cached(["chat", messages, max_tokens], lambda: llm().create_chat_completion(
        messages=messages, temperature=0.0, max_tokens=max_tokens)["choices"][0]["message"]["content"].strip())

_LID = {}
def lid(text):
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

print("UD files:", ", ".join(k.split("/")[1] for k in SHA256))

In [ ]:
import pyarrow.parquet as pq
from transformers import AutoTokenizer
TUMLU_DIR = Path(os.environ.get("TUMLU_DIR", "data/tumlu-mini")).expanduser()
EXPECTED = {"dev": "cd73f20ccd71b24c37114410868ba9db9aafcd8f488456885d11ec561618560a",
            "test": "015331251fb36819de4468909acc2d1517ffd620e46af22f9de928120ae411c2"}   # the book's audit
rows = {}
for split, h in EXPECTED.items():
    p = next(TUMLU_DIR.rglob(f"{split}-00000-of-00001.parquet"))
    assert sha256(p) == h, f"{p} is not the audited file"
    rows[split] = [r for r in pq.read_table(p).to_pylist()]
valid = {s: [r for r in rs if isinstance(r["choices"], list) and len(r["choices"]) == 4 and all(c and c.strip() for c in r["choices"])]
         for s, rs in rows.items()}
print({s: f"{len(valid[s])} valid of {len(rows[s])}" for s in rows})
INSTRUCTION = "Select the correct option. Reply with only its letter."      # as in the book's run
qtok = AutoTokenizer.from_pretrained(os.environ["QWEN_TOKENIZER"])
LETTERS = [qtok.encode(l, add_special_tokens=False)[0] for l in "ABCD"]
def letter_logprobs(question, choices):
    """Log-probabilities of the next tokens A, B, C, D after the chat-formatted prompt."""
    text = INSTRUCTION + "\n\n" + question + "\n" + "\n".join(f"{l}. {c}" for l, c in zip("ABCD", choices))
    prompt = qtok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
    def run():
        import numpy as np, llama_cpp
        m = llm(); ids = m.tokenize(prompt.encode("utf-8"), add_bos=False, special=True)
        m.reset(); m.eval(ids)
        ptr = llama_cpp.llama_get_logits_ith(m._ctx.ctx, -1)                 # logits after the last prompt token
        logits = np.ctypeslib.as_array(ptr, shape=(m.n_vocab(),)).astype(np.float64)
        lp = logits - logits.max(); lp = lp - np.log(np.exp(lp).sum())
        return [float(lp[t]) for t in LETTERS]
    return cached(["tumlu", prompt], run)
def rotate(choices, gold, k):
    """Cyclic shift by k: the option at position i moves to position (i + k) mod 4."""
    new = [None] * 4
    for i, c in enumerate(choices):
        new[(i + k) % 4] = c
    return new, "ABCD"[("ABCD".index(gold) + k) % 4]

# 1. Order sensitivity on the 40 development items: the same question under four cyclic rotations.
dev = valid["dev"]; per_rot, picks = [], []
for k in range(4):
    ok, chosen = 0, []
    for r in dev:
        ch, g = rotate(r["choices"], r["answer"], k)
        lp = letter_logprobs(r["question"], ch); pred = "ABCD"[max(range(4), key=lambda i: lp[i])]
        ok += pred == g; chosen.append(ch["ABCD".index(pred)])
    per_rot.append(ok); picks.append(chosen)
stable = sum(len({picks[k][i] for k in range(4)}) == 1 for i in range(len(dev)))
print(f"dev accuracy by rotation: {per_rot} of {len(dev)}; same option content chosen under all four rotations: {stable}/{len(dev)}")
letters = collections.Counter()
for k in range(4):
    for r in dev:
        ch, g = rotate(r["choices"], r["answer"], k); lp = letter_logprobs(r["question"], ch)
        letters["ABCD"[max(range(4), key=lambda i: lp[i])]] += 1
print("predicted letters over the 160 dev prompts:", dict(sorted(letters.items())))
def debiased(r):
    """Sum each option's log-probability over the four rotations (position-averaged score)."""
    total = [0.0] * 4
    for k in range(4):
        ch, _ = rotate(r["choices"], r["answer"], k); lp = letter_logprobs(r["question"], ch)
        for i in range(4):
            total[i] += lp[(i + k) % 4]
    return "ABCD"[max(range(4), key=lambda i: total[i])]
dev_plain = per_rot[0]; dev_avg = sum(debiased(r) == r["answer"] for r in dev)
policy = "rotation-averaged" if dev_avg > dev_plain else "original order"
print(f"dev: original order {dev_plain}/{len(dev)}, rotation-averaged {dev_avg}/{len(dev)} -> frozen policy: {policy}")

# 2. The frozen policy on the 797 valid test questions.
test = valid["test"]
if policy == "original order":
    preds = ["ABCD"[max(range(4), key=lambda i: letter_logprobs(r["question"], r["choices"])[i])] for r in test]
else:
    preds = [debiased(r) for r in test]
k = sum(p == r["answer"] for p, r in zip(preds, test)); lo, hi = wilson(k, len(test))
dev_major = collections.Counter(r["answer"] for r in dev).most_common(1)[0][0]
km = sum(r["answer"] == dev_major for r in test)
print(f"test ({policy}): {k}/{len(test)} = {k / len(test):.1%} (95% CI {lo:.1%}-{hi:.1%}); "
      f"always '{dev_major}' (dev majority): {km}/{len(test)} = {km / len(test):.1%}; uniform chance 25%")
print("gold letters:", dict(sorted(collections.Counter(r["answer"] for r in test).items())),
      "| predicted:", dict(sorted(collections.Counter(preds).items())))

## Worked solution

Output of `experiments/ch15_exercises/run_ch15_exercises.py`:

```
{'dev': '40 valid of 40', 'test': '797 valid of 799'}
dev accuracy by rotation: [12, 18, 15, 11] of 40; same option content chosen under all four rotations: 4/40
predicted letters over the 160 dev prompts: {'A': 41, 'B': 80, 'C': 29, 'D': 10}
dev: original order 12/40, rotation-averaged 9/40 -> frozen policy: original order
test (original order): 230/797 = 28.9% (95% CI 25.8%-32.1%); always 'D' (dev majority): 304/797 = 38.1%; uniform chance 25%
gold letters: {'A': 175, 'B': 170, 'C': 148, 'D': 304} | predicted: {'A': 247, 'B': 351, 'C': 132, 'D': 67}
```

**Option order changes the selected content.** The four cyclic rotations yield 12, 18, 15 and 11 correct answers out of 40 development questions. Only four questions retain the same selected content under all rotations. B is selected in 80 of 160 prompts, compared with ten D selections. This demonstrates order sensitivity in the tested protocol, not a general measure of the model's knowledge.

**Choose the policy on development data.** Combining each option's log-probabilities across rotations gives nine correct answers, against twelve in the original order. This aggregation tests one way to reduce order sensitivity; it does not guarantee removal of bias. The original-order policy is then applied to the 797 valid test questions, giving 230 correct (28.9%; Wilson 95% interval 25.8–32.1%). The interval assumes independent items.

The chapter's separate run scores 226/797. The exercise changes numerical precision and inference implementation, so the difference cannot be attributed to quantisation alone.

**Show the baseline and label distribution.** D is the most frequent development answer. Always selecting D scores 304/797 (38.1%) on the test subset, above the model's point estimate and the 25% uniform-choice expectation. Report both baselines. With only 40 development items, policy selection is also uncertain.

## Your turn — extension

1. Use all 24 permutations of the options for ten development questions and measure how often the chosen content changes.
2. Score the full answer text instead of the letter (sum of log-probabilities of the option string) and compare.